# Late fusion

Average the saved text and subreddit test predictions. Run the two baselines first.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from sklearn.metrics import mean_squared_error, r2_score

TEXT = Path("../data/text_baseline/predictions.csv")
SUBREDDIT = Path("../data/subreddit_baseline/predictions.csv")
OUTPUT = Path("../data/late_fusion")

## 1. Match and average the predictions

Match by user, trait, and outer fold. Give the two models equal weight.

In [ ]:
keys = ["author", "trait", "outer_fold"]
text = pd.read_csv(TEXT)[keys + ["true", "text_prediction"]]
subreddit = pd.read_csv(SUBREDDIT)[keys + ["true", "subreddit_prediction"]]

paired = text.merge(subreddit, on=keys, validate="one_to_one",
                    suffixes=("", "_subreddit"))
if (paired.empty or len(paired) != len(text) or len(paired) != len(subreddit)
        or paired.duplicated(["author", "trait"]).any()):
    raise ValueError("Expected one matching test prediction per user and trait.")
if not paired["true"].eq(paired["true_subreddit"]).all():
    raise ValueError("The files disagree on the true scores.")

paired["late_fusion_prediction"] = (
    paired["text_prediction"] + paired["subreddit_prediction"]
) / 2

## 2. Compare the models

A negative RMSE change means late fusion made smaller errors than text alone.

In [ ]:
metrics = []
for trait, rows in paired.groupby("trait"):
    actual = rows["true"]
    for model in ["text", "subreddit", "late_fusion"]:
        predicted = rows[f"{model}_prediction"]
        correlation = np.nan
        if actual.nunique() > 1 and predicted.nunique() > 1:
            correlation = pearsonr(actual, predicted)[0]
        metrics.append({
            "trait": trait,
            "model": model,
            "rmse": mean_squared_error(actual, predicted) ** 0.5,
            "pearson_r": correlation,
            "r2": r2_score(actual, predicted) if actual.nunique() > 1 else np.nan,
        })

metrics = pd.DataFrame(metrics)
comparison = metrics.pivot(index="trait", columns="model", values="rmse")
comparison = comparison[["text", "subreddit", "late_fusion"]]
comparison["rmse_change"] = comparison["late_fusion"] - comparison["text"]

## 3. Save the results

In [ ]:
OUTPUT.mkdir(parents=True, exist_ok=True)
paired.drop(columns="true_subreddit").to_csv(OUTPUT / "predictions.csv", index=False)
metrics.to_csv(OUTPUT / "metrics.csv", index=False)
comparison.to_csv(OUTPUT / "comparison.csv")
display(comparison.round(3))
display(metrics.round(3))